In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.bigru_tcn_wrapper import BiGRU_TCNModel
from src.trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "BiGRU_TCN"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


In [2]:
import math
import time

scorer = VolatilityScorer()

look_back = 14
kernel_size = 2
num_convs = 2
min_levels = math.ceil(
    math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
)
tcn_levels = max(min_levels, 2)

best_model = BiGRU_TCNModel(
    look_back=look_back,
    bigru_hidden=64,
    tcn_channels=32,
    tcn_levels=tcn_levels,
    kernel_size=kernel_size,
    num_convs=num_convs,
    dense_units=64,
    num_dense_layers=2,
    dropout=0.2,
    learning_rate=1e-3,
    weight_decay=1e-6,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    fine_tune_epochs=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time
print(f">> fit OK en {elapsed_time:.2f} s")
print("   diagnostics:", best_model.get_diagnostics())

n_oos = min(2, len(y_test))
y_test_20 = y_test[-n_oos:]
rs_test_20 = rs_test[-n_oos:]

t0_oos = time.time()
preds_test_20 = best_model.forecast(y_test_20, rs_seq=rs_test_20, debug=False)
oos_time = time.time() - t0_oos

print(
    f">> forecast (últimos {n_oos} días, fine_tune_epochs=1) OK en {oos_time:.2f} s "
    f"| shape={preds_test_20.shape}"
)

>> fit OK en 77.11 s
   diagnostics: {'converged': True, 'epochs_run': 51}
>> forecast (últimos 2 días, fine_tune_epochs=1) OK en 141.06 s | shape=(2,)


In [3]:
import numpy as np
from src.utils import trainer_toolkit as tt

history = best_model.get_history()
diagnostics = best_model.get_diagnostics()

train_loss = np.asarray(history.get("train_loss", []), dtype=float)
val_loss = np.asarray(history.get("val_loss", []), dtype=float)

assert len(train_loss) > 0 and len(val_loss) > 0, "El historial de entrenamiento está vacío."
assert np.isfinite(train_loss).all(), "train_loss contiene NaN o Inf."
assert np.isfinite(val_loss).all(), "val_loss contiene NaN o Inf."

print(
    f"Épocas ejecutadas: {diagnostics.get('epochs_run', len(train_loss))} "
    f"| Convergió: {diagnostics.get('converged')}"
)
print(
    f"Train QLIKE inicial -> final/mínimo: {train_loss[0]:.6f} -> {train_loss[-1]:.6f} "
    f"(min={train_loss.min():.6f})"
)
print(
    f"Val   QLIKE inicial -> final/mínimo: {val_loss[0]:.6f} -> {val_loss[-1]:.6f} "
    f"(min={val_loss.min():.6f})"
)

max_epochs_plotted = len(train_loss)

tt.plot_learning_curve(
    history=history,
    title="Curva de Aprendizaje (BiGRU-TCN)",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14,
    ylab_pad=14,
    line_width=2.0,
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
    ylim=(0, 5),
    xlim=(0, max(10, max_epochs_plotted)),
    y_step=0.5,
    x_step=5,
    legend_bbox=(0.95, 0.95),
)

Épocas ejecutadas: 51 | Convergió: True
Train QLIKE inicial -> final/mínimo: 6.905363 -> 0.930379 (min=0.846193)
Val   QLIKE inicial -> final/mínimo: 6.072059 -> 0.429579 (min=0.360471)


In [4]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_scaled_target = rs_train * best_model.target_scale

x_windows = np.lib.stride_tricks.sliding_window_view(y_train_scaled, window_shape=look_back)
x_windows = x_windows[:-1, :, None].astype(np.float32)
y_targets = rs_train_scaled_target[look_back:].astype(np.float32)

train_eval_dataset = TensorDataset(
    torch.tensor(x_windows, dtype=torch.float32),
    torch.tensor(y_targets, dtype=torch.float32),
)
train_eval_loader = DataLoader(
    train_eval_dataset, batch_size=best_model.batch_size, shuffle=False
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

dates_aligned = df_train.index[look_back : look_back + n_preds]
rs_volatilidad_aligned = np.sqrt(
    np.maximum(rs_train[look_back : look_back + n_preds], 0.0)
)
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (BiGRU-TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad BiGRU-TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

In [ ]:
import math
import optuna


def bigru_tcn_search_space(trial: optuna.Trial) -> dict:
    look_back = trial.suggest_categorical("look_back", [7, 14, 21, 28])
    kernel_size = trial.suggest_categorical("kernel_size", [2, 3])
    num_convs = trial.suggest_categorical("num_convs", [1, 2, 3])

    min_levels = math.ceil(
        math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
    )
    max_levels = min(min_levels + 2, 5)

    return {
        "look_back": look_back,
        "bigru_hidden": trial.suggest_categorical("bigru_hidden", [32, 64, 128]),
        "tcn_channels": trial.suggest_categorical("tcn_channels", [32, 64, 128]),
        "tcn_levels": trial.suggest_int("tcn_levels", min_levels, max_levels),
        "kernel_size": kernel_size,
        "num_convs": num_convs,
        "dense_units": trial.suggest_categorical("dense_units", [32, 64, 128, 256]),
        "num_dense_layers": trial.suggest_categorical("num_dense_layers", [1, 2]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=BiGRU_TCNModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=bigru_tcn_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)